# Model Training

Trains a RandomForestRegressor on the feature store data to predict `total_demand`.



In [0]:
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from mlflow.models import infer_signature



In [0]:
# Config
CATALOG = "oil_gas_demo"
FEATURE_TABLE = f"{CATALOG}.gold.demand_feature_store"
LABEL_TABLE = f"{CATALOG}.gold.demand_labels"
TRAIN_TABLE = f"{CATALOG}.ml_model.training_data"
VAL_TABLE = f"{CATALOG}.ml_model.validation_data"
TEST_TABLE = f"{CATALOG}.ml_model.testing_data"
MODEL_NAME = f"{CATALOG}.ml_model.random_forest_demand"
EXPERIMENT_NAME = "/Shared/demand_forecasting_experiment"

# Time-series feature columns that may have nulls
TS_FEATURE_COLS = ["demand_lag_1", "demand_lag_7", "rolling_avg_7"]

# Columns to drop from model features
DROP_COLS = ["feature_uid", "transaction_date", "total_demand", "product_name", "destination_city"]


In [0]:
# Load feature and label tables
features_df = spark.table(FEATURE_TABLE)
labels_df = spark.table(LABEL_TABLE)

print(f"Features: {features_df.count():,} rows")
print(f"Labels: {labels_df.count():,} rows")
features_df.printSchema()

In [0]:
# Join features + labels on feature_uid
# Rename label key to avoid duplicate-column error
labels_renamed = labels_df.withColumnRenamed("feature_uid", "label_feature_uid")

joined = (
    features_df.join(labels_renamed, features_df.feature_uid == labels_renamed.label_feature_uid, "inner")
    .drop("label_feature_uid")
)

print(f"Joined: {joined.count():,} rows")
display(joined.limit(5))

In [0]:
# Handle missing values in time-series feature columns
for col_name in TS_FEATURE_COLS:
    if col_name in joined.columns:
        joined = joined.fillna({col_name: 0.0})

print("Missing values handled.")

In [0]:
# Chronological date-based split: 64% train, 16% validation, 20% test
# Get sorted unique dates
dates = [row.transaction_date for row in joined.select("transaction_date").distinct().orderBy("transaction_date").collect()]

n_dates = len(dates)
train_end = int(n_dates * 0.64)
val_end = int(n_dates * 0.80)

train_dates = set(dates[:train_end])
val_dates = set(dates[train_end:val_end])
test_dates = set(dates[val_end:])

train_df = joined.filter(F.col("transaction_date").isin(list(train_dates)))
val_df = joined.filter(F.col("transaction_date").isin(list(val_dates)))
test_df = joined.filter(F.col("transaction_date").isin(list(test_dates)))

print(f"Train dates: {dates[0]} to {dates[train_end - 1]} ({len(train_dates)} dates)")
print(f"Val dates:   {dates[train_end]} to {dates[val_end - 1]} ({len(val_dates)} dates)")
print(f"Test dates:  {dates[val_end]} to {dates[-1]} ({len(test_dates)} dates)")
print(f"Train: {train_df.count():,} | Val: {val_df.count():,} | Test: {test_df.count():,}")

In [0]:
# Save split data as Delta tables (non-encoded, original)
# train_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TRAIN_TABLE)
# val_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(VAL_TABLE)
# test_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TEST_TABLE)

# print(f"Saved: {TRAIN_TABLE}, {VAL_TABLE}, {TEST_TABLE}")

In [0]:
# Convert to pandas for sklearn
train_pd = train_df.toPandas()
val_pd = val_df.toPandas()
test_pd = test_df.toPandas()

print(f"Train: {len(train_pd):,} | Val: {len(val_pd):,} | Test: {len(test_pd):,}")

In [0]:
# OneHotEncode product_name and destination_city
# Fit ONLY on training data, then transform val and test
cat_cols = ["product_name", "destination_city"]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_pd[cat_cols])

# Transform all splits
train_encoded = encoder.transform(train_pd[cat_cols])
val_encoded = encoder.transform(val_pd[cat_cols])
test_encoded = encoder.transform(test_pd[cat_cols])

# Get encoded column names
encoded_cols = encoder.get_feature_names_out(cat_cols).tolist()

# Create encoded DataFrames
train_encoded_df = pd.DataFrame(train_encoded, columns=encoded_cols, index=train_pd.index)
val_encoded_df = pd.DataFrame(val_encoded, columns=encoded_cols, index=val_pd.index)
test_encoded_df = pd.DataFrame(test_encoded, columns=encoded_cols, index=test_pd.index)

# Drop original cat cols and concatenate encoded
drop_cols_present = [c for c in DROP_COLS if c in train_pd.columns]

X_train = pd.concat([train_pd.drop(columns=drop_cols_present), train_encoded_df], axis=1)
X_val = pd.concat([val_pd.drop(columns=drop_cols_present), val_encoded_df], axis=1)
X_test = pd.concat([test_pd.drop(columns=drop_cols_present), test_encoded_df], axis=1)

y_train = train_pd["total_demand"]
y_val = val_pd["total_demand"]
y_test = test_pd["total_demand"]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")
print(f"Feature columns: {list(X_train.columns)}")

In [0]:
# Train RandomForestRegressor
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

print("Model trained.")

In [0]:
# Evaluate on validation and test sets
val_preds = model.predict(X_val)
test_preds = model.predict(X_test)

val_mae = mean_absolute_error(y_val, val_preds)
val_mse = mean_squared_error(y_val, val_preds)
val_rmse = mean_squared_error(y_val, val_preds) ** 0.5
val_r2 = r2_score(y_val, val_preds)

test_mae = mean_absolute_error(y_test, test_preds)
test_mse = mean_squared_error(y_test, test_preds)
test_rmse = mean_squared_error(y_test, test_preds) ** 0.5
test_r2 = r2_score(y_test, test_preds)

print("--- Validation Metrics ---")
print(f"MAE:  {val_mae:.2f}")
print(f"MSE:  {val_mse:.2f}")
print(f"RMSE: {val_rmse:.2f}")
print(f"R2:   {val_r2:.4f}")

print("--- Test Metrics ---")
print(f"MAE:  {test_mae:.2f}")
print(f"MSE:  {test_mse:.2f}")
print(f"RMSE: {test_rmse:.2f}")
print(f"R2:   {test_r2:.4f}")

In [0]:
# MLflow tracking
mlflow.set_experiment(EXPERIMENT_NAME)

signature = infer_signature(X_train.head(100), model.predict(X_train.head(100)))

with mlflow.start_run() as run:
    run_id = run.info.run_id

    # Log parameters
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("n_jobs", -1)

    # Log record counts
    mlflow.log_param("train_rows", len(train_pd))
    mlflow.log_param("val_rows", len(val_pd))
    mlflow.log_param("test_rows", len(test_pd))

    # Log validation metrics
    mlflow.log_metric("val_mae", val_mae)
    mlflow.log_metric("val_mse", val_mse)
    mlflow.log_metric("val_rmse", val_rmse)
    mlflow.log_metric("val_r2", val_r2)

    # Log test metrics
    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log fitted encoder as artifact
    joblib.dump(encoder, "/tmp/encoder.joblib")
    mlflow.log_artifact("/tmp/encoder.joblib", artifact_path="encoder")

    # Log model with signature
    model_info = mlflow.sklearn.log_model(
        model,
        name="random_forest_demand_model",
        signature=signature,
        input_example=X_train.head(3),
    )

print(f"MLflow run: {run_id}")
print(f"Model URI: {model_info.model_uri}")

In [0]:
# Register model in Unity Catalog
client = mlflow.tracking.MlflowClient()

result = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=MODEL_NAME,
)

print(f"Registered: {MODEL_NAME} version {result.version}")

In [0]:
# Select best run by lowest validation MAE (ascending order)
best_run = (
    client.search_runs(
        experiment_ids=[client.get_experiment_by_name(EXPERIMENT_NAME).experiment_id],
        order_by=["metrics.val_mae ASC"],
        max_results=1,
    )
    [0]
)

best_run_id = best_run.info.run_id
best_val_mae = best_run.data.metrics["val_mae"]

print(f"Best run: {best_run_id} | val_MAE: {best_val_mae:.2f}")

In [0]:
# Assign alias "challenger" to the newly trained model version.
# In the champion/challenger workflow:
#   - "champion"  alias -> current production model (set by the evaluation notebook)
#   - "challenger" alias -> new candidate model (set here after each training run)
# The evaluation notebook compares both and auto-promotes the winner.
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="challenger",
    version=result.version,
)

print(f"Alias 'challenger' assigned to {MODEL_NAME} version {result.version}")

In [0]:
# Load model by alias
loaded_model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@challenger")

print("Model loaded via alias 'challenger'.")

In [0]:
# Create final prediction table
all_pd = pd.concat([train_pd, val_pd, test_pd], ignore_index=True)

# Re-encode all data using the fitted encoder
all_encoded = encoder.transform(all_pd[cat_cols])
all_encoded_df = pd.DataFrame(all_encoded, columns=encoded_cols, index=all_pd.index)
all_drop = [c for c in DROP_COLS if c in all_pd.columns]
X_all = pd.concat([all_pd.drop(columns=all_drop), all_encoded_df], axis=1)

# Predict
all_pd["predicted_demand"] = model.predict(X_all)

# Select final columns
pred_cols = ["feature_uid", "transaction_date", "product_name", "destination_city", "total_demand", "predicted_demand"]
predictions_pd = all_pd[[c for c in pred_cols if c in all_pd.columns]]

# Save as Spark table
pred_table = f"{CATALOG}.ml_model.predictions"
spark.createDataFrame(predictions_pd).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(pred_table)

print(f"Predictions saved to: {pred_table} ({len(predictions_pd):,} rows)")
display(spark.table(pred_table).limit(10))

In [0]:
# Final summary
# print("=" * 50)
# print("MODEL TRAINING COMPLETE")
# print("=" * 50)
# print(f"Training rows:   {len(train_pd):,}")
# print(f"Validation rows: {len(val_pd):,}")
# print(f"Testing rows:    {len(test_pd):,}")
# print()
# print("--- Validation Metrics ---")
# print(f"MAE:  {val_mae:.2f}")
# print(f"MSE:  {val_mse:.2f}")
# print(f"RMSE: {val_rmse:.2f}")
# print(f"R2:   {val_r2:.4f}")
# print()
# print("--- Test Metrics ---")
# print(f"MAE:  {test_mae:.2f}")
# print(f"MSE:  {test_mse:.2f}")
# print(f"RMSE: {test_rmse:.2f}")
# print(f"R2:   {test_r2:.4f}")
# print()
# print(f"Registered model: {MODEL_NAME}")
# print(f"Alias: dev")
# print(f"MLflow run: {run_id}")
# print(f"Best run (lowest val_MAE): {best_run_id}")
# print("=" * 50)

Champion and challanger 

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

MODEL_NAME = "oil_gas_demo.ml_model.random_forest_demand"

# Get the existing Unity Catalog model
registered_model = client.get_registered_model(MODEL_NAME)

print("Model:", registered_model.name)
print("Existing aliases:", registered_model.aliases)

# Get all registered versions
versions = client.search_model_versions(
    filter_string=f"name = '{MODEL_NAME}'"
)

for version in versions:
    print("\n-----------------------------")
    print("Version:", version.version)
    print("Run ID:", version.run_id)
    print("Aliases:", version.aliases)

    # Display the metrics logged by the training run
    if version.run_id:
        run = client.get_run(version.run_id)

        print("Metrics:")
        for metric_name, metric_value in run.data.metrics.items():
            print(f"  {metric_name}: {metric_value}")

In [0]:
MODEL_NAME = "oil_gas_demo.ml_model.random_forest_demand"

champion_uri = f"models:/{MODEL_NAME}@champion"
challenger_uri = f"models:/{MODEL_NAME}@challenger"

champion_model = mlflow.pyfunc.load_model(champion_uri)
challenger_model = mlflow.pyfunc.load_model(challenger_uri)

print("Champion model loaded successfully")
print("Challenger model loaded successfully")